In [1]:
import json
from typing import Dict, Any, List

def load_global_entities_sorted_by_degree(path: str) -> List[Dict[str, Any]]:
    """
    Load entities from JSON file, filter scope == 'global',
    and sort by total_degree descending.
    """
    with open(path, "r", encoding="utf-8") as f:
        data: Dict[str, Dict[str, Any]] = json.load(f)

    global_entities = [
        ent for ent in data.values()
        if ent.get("scope") == "global"
    ]

    global_entities_sorted = sorted(
        global_entities,
        key=lambda x: x.get("total_degree", 0),
        reverse=True
    )

    return global_entities_sorted


In [2]:
entities = load_global_entities_sorted_by_degree("data/knowledge_graph/entity_info_refined.json")

In [3]:
entities

[{'id': 'ent_2cfa50f104b1',
  'name': 'Berlin',
  'type': 'Character',
  'scope': 'global',
  'description': "A lone individual driving a car, smoking cigarettes, and navigating through traffic towards an intersection.\nA person found waking up on a new mattress in the middle of a derelict house with bare wooden boards and no furniture.\nA character who searches the garage, reacts with frustration, and walks away while a voice begins to intrude.\nA man who listens to a police scanner, eats an unusual breakfast, and struggles with smoking while unloading items from his leather bag.\nA forensic officer or investigator who arrives at the crime scene, reports on the cause of death, and interacts with Ross while wearing surgical gloves.\nA character who engages Ross in conversation, asking about past activities and the missing kid, showing concern or curiosity about ongoing events.\nA character who examines the trash sack after Ross, comments on the severed hand possibly belonging to a woma

In [4]:
import pickle

def load_graph(graph_path: str):
    with open(graph_path, "rb") as f:
        G = pickle.load(f)
    return G


In [5]:
graph_path = "data/knowledge_graph/graph_nx.pkl"

G = load_graph(graph_path)

In [6]:
edges = []
node_id = "ent_bda73fd59e19"

context = []


# 出边
for _, v, k, data in G.out_edges(node_id, keys=True, data=True):
    edges.append({
        "direction": "out",
        "src": node_id,
        "dst": v,
        "key": k,
        "relation": data,
    })
    if data["relation_type"] not in ["speaks_to", "favorable_toward", "unfavorable_toward"]:
        context.append(
            # f"({data["subject"]}, {data["relation_name"]}, {data["object"]})"
            data["description"]
        )

# 入边
for u, _, k, data in G.in_edges(node_id, keys=True, data=True):
    edges.append({
        "direction": "in",
        "src": u,
        "dst": node_id,
        "key": k,
        "relation": data,
    })
    if data["relation_type"] not in ["speaks_to", "favorable_toward", "unfavorable_toward"]:
        context.append(
            # f"({data["subject"]}, {data["relation_name"]}, {data["object"]})"
            data["description"]
        )


In [35]:
G.nodes[node_id]

{'id': 'ent_bda73fd59e19',
 'name': 'Helena Robertson',
 'type': 'Character',
 'scope': 'global',
 'description': "A young woman with delicate features and dark eyes who opens the door for Ross and Berlin, appearing shy during introductions.\nA woman being questioned about her memories of spending time with Amber on the afternoon she left, appearing nervous during the conversation.\nA blind woman who recalls a man urging her to hurry due to snow and becomes uncomfortable when Berlin examines her hands.\nA woman who walks through the corridor unaware of the tension between Ross and Berlin and answers a question about her teaching subject, music composition and cello.\nA woman rehearsing music in an empty gymnasium, playing Elgar, who becomes aware of Berlin's presence and interacts with him.\nA woman providing testimony or recounting an observation about a car's sound, engaging in dialogue with Berlin.\nA woman who remains at the door, engages in conversation with Berlin, recalls someon

In [7]:
edges[0]

{'direction': 'out',
 'src': 'ent_bda73fd59e19',
 'dst': 'ent_ef5da3c10ace',
 'key': 'rel_a9719b0e32d3',
 'relation': {'id': 'rel_a9719b0e32d3',
  'relation_type': 'performs',
  'subject': 'Helena Robertson',
  'object': 'Helena Robertson opens the door for Ross and Berlin',
  'subject_id': 'ent_bda73fd59e19',
  'object_id': 'ent_ef5da3c10ace',
  'relation_name': 'opens the door for Ross and Berlin',
  'description': 'Helena Robertson is directly stated to open the door when the footsteps arrive.',
  'persistence': 'momentary',
  'base_chapter': 'scene_32',
  'chapter_key': 'scene_32_part_1',
  'source_chapters': ['scene_32_part_1']}}

In [60]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Set, Tuple, Literal

import math
import statistics

import networkx as nx
import matplotlib.pyplot as plt


CentralityMetric = Literal["pagerank", "total_degree", "closeness", "betweenness"]


# ----------------------------
# Node filtering
# ----------------------------

def _node_attr(G: nx.Graph, n: Any, key: str, default: Any = None) -> Any:
    return (G.nodes.get(n, {}) or {}).get(key, default)


def get_nodes_by_type_and_scope(
    G: nx.Graph,
    *,
    node_type: Optional[str] = None,
    scope: Optional[str] = "global",
    type_key: str = "type",
    scope_key: str = "scope",
) -> List[Any]:
    """
    Flexible node selection.

    - If node_type is None: do NOT filter by type.
    - If scope is None: do NOT filter by scope.
    - Default scope="global" matches your original behavior.
      If you want "all scopes", pass scope=None.
    """
    out: List[Any] = []
    for n in G.nodes:
        if node_type is not None and _node_attr(G, n, type_key) != node_type:
            continue
        if scope is not None and _node_attr(G, n, scope_key) != scope:
            continue
        out.append(n)
    return out


# ----------------------------
# Edge filtering (relation_type)
# ----------------------------

def filter_graph_by_relation_type(
    G: nx.Graph,
    *,
    exclude_relation_types: Optional[Set[str]] = None,
    include_relation_types: Optional[Set[str]] = None,
    # Your edge payload path: data["relation"]["relation_type"]
    relation_field: str = "relation",
    relation_type_field: str = "relation_type",
    keep_isolated_nodes: bool = True,
) -> nx.Graph:
    """
    Build a subgraph copy that filters edges based on relation_type.

    - include_relation_types: keep only edges with relation_type in the set
    - exclude_relation_types: drop edges with relation_type in the set
    - if neither provided: returns a full copy

    Missing relation_type:
    - if include_relation_types is set: drop
    - otherwise: keep
    """
    if include_relation_types and exclude_relation_types:
        raise ValueError("Provide only one of include_relation_types or exclude_relation_types.")

    H = G.__class__()  # preserve graph class
    if keep_isolated_nodes:
        H.add_nodes_from(G.nodes(data=True))

    def keep_edge(rtype: Optional[str]) -> bool:
        if include_relation_types is not None:
            return rtype in include_relation_types
        if exclude_relation_types is not None:
            return rtype not in exclude_relation_types
        return True

    if isinstance(G, (nx.MultiGraph, nx.MultiDiGraph)):
        for u, v, k, data in G.edges(keys=True, data=True):
            data = data or {}
            rel = (data.get(relation_field, {}) or {}) if isinstance(data, dict) else {}
            rtype = rel.get(relation_type_field, None) if isinstance(rel, dict) else None

            if include_relation_types is not None and rtype is None:
                continue
            if not keep_edge(rtype):
                continue

            H.add_edge(u, v, key=k, **data)
    else:
        for u, v, data in G.edges(data=True):
            data = data or {}
            rel = (data.get(relation_field, {}) or {}) if isinstance(data, dict) else {}
            rtype = rel.get(relation_type_field, None) if isinstance(rel, dict) else None

            if include_relation_types is not None and rtype is None:
                continue
            if not keep_edge(rtype):
                continue

            H.add_edge(u, v, **data)

    return H


# ----------------------------
# MultiGraph -> simple Graph conversion
# ----------------------------

def _to_simple_graph(
    G: nx.Graph,
    *,
    directed: Optional[bool] = None,
    edge_weight_attr: Optional[str] = None,
    multiedge_weight_mode: Literal["count", "sum"] = "count",
) -> nx.Graph:
    """
    Collapse parallel edges for MultiGraph/MultiDiGraph into a simple Graph/DiGraph.

    - Always creates/updates edge attribute "weight"
    - multiedge_weight_mode:
        - "count": weight = number of parallel edges
        - "sum":   weight = sum(edge_weight_attr) if present else count
    """
    if directed is None:
        directed = G.is_directed()

    SG = nx.DiGraph() if directed else nx.Graph()
    SG.add_nodes_from(G.nodes(data=True))

    if isinstance(G, (nx.MultiGraph, nx.MultiDiGraph)):
        for u, v, k, data in G.edges(keys=True, data=True):
            data = data or {}
            w = 1.0
            if multiedge_weight_mode == "sum" and edge_weight_attr is not None:
                if isinstance(data, dict) and edge_weight_attr in data and data[edge_weight_attr] is not None:
                    try:
                        w = float(data[edge_weight_attr])
                    except Exception:
                        w = 1.0

            if SG.has_edge(u, v):
                if multiedge_weight_mode == "count":
                    SG[u][v]["weight"] = float(SG[u][v].get("weight", 1.0)) + 1.0
                else:
                    SG[u][v]["weight"] = float(SG[u][v].get("weight", 0.0)) + w
            else:
                if multiedge_weight_mode == "count":
                    SG.add_edge(u, v, weight=1.0)
                else:
                    SG.add_edge(u, v, weight=w)
    else:
        SG = G.copy()
        for u, v, data in SG.edges(data=True):
            if isinstance(data, dict) and "weight" not in data:
                if edge_weight_attr is not None and edge_weight_attr in data and data[edge_weight_attr] is not None:
                    try:
                        data["weight"] = float(data[edge_weight_attr])
                    except Exception:
                        data["weight"] = 1.0
                else:
                    data["weight"] = 1.0

    return SG


# ----------------------------
# Centrality computation with optional edge filtering
# ----------------------------

def compute_centrality(
    G: nx.Graph,
    metric: CentralityMetric,
    *,
    # edge filter
    exclude_relation_types: Optional[Set[str]] = None,
    include_relation_types: Optional[Set[str]] = None,
    # pagerank
    pagerank_alpha: float = 0.85,
    # shortest-path distance weight name (distance, not strength). Usually None.
    distance_weight: Optional[str] = None,
    # collapsing policy for multigraph -> simple graph
    multiedge_weight_mode: Literal["count", "sum"] = "count",
    # numeric edge attribute you may want to sum when collapsing
    edge_weight_attr: Optional[str] = None,
) -> Dict[Any, float]:
    """
    Compute centrality dict: node -> score.

    Choices:
      - pagerank: computed on collapsed simple graph with edge strength "weight"
      - total_degree: computed on original (filtered) graph (multiedges count)
      - closeness/betweenness: computed on collapsed simple graph (avoid multiedge path artifacts)
    """
    metric = metric.lower()  # type: ignore

    # Filter edges first
    if include_relation_types is not None or exclude_relation_types is not None:
        G = filter_graph_by_relation_type(
            G,
            include_relation_types=include_relation_types,
            exclude_relation_types=exclude_relation_types,
            keep_isolated_nodes=True,
        )

    if metric == "total_degree":
        if G.is_directed():
            return {n: float(G.in_degree(n) + G.out_degree(n)) for n in G.nodes}
        return {n: float(G.degree(n)) for n in G.nodes}

    if metric == "pagerank":
        SG = _to_simple_graph(
            G,
            directed=G.is_directed(),
            edge_weight_attr=edge_weight_attr,
            multiedge_weight_mode=multiedge_weight_mode,
        )
        return nx.pagerank(SG, alpha=pagerank_alpha, weight="weight")

    if metric in ("closeness", "betweenness"):
        SG = _to_simple_graph(
            G,
            directed=G.is_directed(),
            edge_weight_attr=edge_weight_attr,
            multiedge_weight_mode=multiedge_weight_mode,
        )
        if metric == "closeness":
            return nx.closeness_centrality(SG, distance=distance_weight)
        return nx.betweenness_centrality(SG, weight=distance_weight, normalized=True)

    raise ValueError(f"Unknown metric: {metric}")


# ----------------------------
# Selection by (optional type, scope, metric, threshold)
# ----------------------------

def filter_nodes_by_centrality(
    G: nx.Graph,
    *,
    metric: CentralityMetric,
    threshold: float,
    node_type: Optional[str] = None,   # None => all types
    scope: Optional[str] = "global",   # None => all scopes
    num_top: Optional[int] = None,
    # edge filter
    exclude_relation_types: Optional[Set[str]] = None,
    include_relation_types: Optional[Set[str]] = None,
    # centrality args
    pagerank_alpha: float = 0.85,
    distance_weight: Optional[str] = None,
    multiedge_weight_mode: Literal["count", "sum"] = "count",
    edge_weight_attr: Optional[str] = None,
) -> List[Tuple[Any, float]]:
    """
    Selection criteria:
      - node_type filter if provided, else all nodes
      - scope filter if provided (default "global"), else all scopes if scope=None
      - compute centrality
      - keep nodes with score > threshold
    """
    candidates = set(get_nodes_by_type_and_scope(G, node_type=node_type, scope=scope))

    scores = compute_centrality(
        G,
        metric,
        exclude_relation_types=exclude_relation_types,
        include_relation_types=include_relation_types,
        pagerank_alpha=pagerank_alpha,
        distance_weight=distance_weight,
        multiedge_weight_mode=multiedge_weight_mode,
        edge_weight_attr=edge_weight_attr,
    )

    selected = [(n, float(scores.get(n, 0.0))) for n in candidates if float(scores.get(n, 0.0)) > float(threshold)]
    selected.sort(key=lambda x: x[1], reverse=True)

    if num_top is not None:
        selected = selected[: int(num_top)]
    return selected


# ----------------------------
# Reporting: stats + histogram + top-k table data
# ----------------------------

@dataclass
class CentralityStats:
    count: int
    min: float
    max: float
    mean: float
    median: float
    stdev: float
    p5: float
    p25: float
    p75: float
    p95: float


def _percentile(vals: List[float], p: float) -> float:
    if not vals:
        return float("nan")
    s = sorted(vals)
    if len(s) == 1:
        return s[0]
    k = (len(s) - 1) * p
    f = math.floor(k)
    c = math.ceil(k)
    if f == c:
        return s[int(k)]
    return s[f] * (c - k) + s[c] * (k - f)


def summarize_scores(scores: Dict[Any, float], nodes: Iterable[Any]) -> CentralityStats:
    vals = [float(scores.get(n, 0.0)) for n in nodes]
    vals = [v for v in vals if not math.isnan(v)]
    if not vals:
        return CentralityStats(
            0, float("nan"), float("nan"), float("nan"), float("nan"), float("nan"),
            float("nan"), float("nan"), float("nan"), float("nan")
        )

    stdev = statistics.pstdev(vals) if len(vals) > 1 else 0.0
    return CentralityStats(
        count=len(vals),
        min=min(vals),
        max=max(vals),
        mean=sum(vals) / len(vals),
        median=statistics.median(vals),
        stdev=stdev,
        p5=_percentile(vals, 0.05),
        p25=_percentile(vals, 0.25),
        p75=_percentile(vals, 0.75),
        p95=_percentile(vals, 0.95),
    )


def export_centrality_report(
    G: nx.Graph,
    *,
    metric: CentralityMetric,
    node_type: Optional[str] = None,   # None => all types
    scope: Optional[str] = "global",   # None => all scopes
    top_k: int = 30,
    # edge filter
    exclude_relation_types: Optional[Set[str]] = None,
    include_relation_types: Optional[Set[str]] = None,
    # centrality args
    pagerank_alpha: float = 0.85,
    distance_weight: Optional[str] = None,
    multiedge_weight_mode: Literal["count", "sum"] = "count",
    edge_weight_attr: Optional[str] = None,
    # histogram
    bins: int = 30,
    show_plot: bool = True,
) -> Dict[str, Any]:
    """
    Report for nodes matching (optional type, optional scope):
      - stats (p5/p25/p75/p95)
      - all_rows (sorted by score desc)
      - top_rows

    Row schema:
      {id, name, type, scope, score, total_degree}
    """
    nodes = get_nodes_by_type_and_scope(G, node_type=node_type, scope=scope)

    scores = compute_centrality(
        G,
        metric,
        exclude_relation_types=exclude_relation_types,
        include_relation_types=include_relation_types,
        pagerank_alpha=pagerank_alpha,
        distance_weight=distance_weight,
        multiedge_weight_mode=multiedge_weight_mode,
        edge_weight_attr=edge_weight_attr,
    )

    stats = summarize_scores(scores, nodes)

    rows: List[Dict[str, Any]] = []
    for n in nodes:
        rows.append(
            {
                "id": n,
                "name": _node_attr(G, n, "name"),
                "type": _node_attr(G, n, "type"),
                "scope": _node_attr(G, n, "scope"),
                "score": float(scores.get(n, 0.0)),
                "total_degree": float(G.degree(n)) if not G.is_directed() else float(G.in_degree(n) + G.out_degree(n)),
            }
        )

    rows.sort(key=lambda r: r["score"], reverse=True)
    top_rows = rows[: int(top_k)]

    if show_plot:
        vals = [r["score"] for r in rows]
        plt.figure()
        plt.hist(vals, bins=bins)
        title_parts = [f"{metric} histogram", f"n={len(vals)}"]
        if scope is not None:
            title_parts.insert(1, f"scope={scope}")
        if node_type is not None:
            title_parts.insert(1, f"type={node_type}")
        if exclude_relation_types:
            title_parts.append(f"excluded_rel_types={len(exclude_relation_types)}")
        if include_relation_types:
            title_parts.append(f"included_rel_types={len(include_relation_types)}")
        plt.title(" | ".join(title_parts))
        plt.xlabel(metric)
        plt.ylabel("count")
        plt.tight_layout()
        plt.show()

    return {
        "metric": metric,
        "node_type": node_type,
        "scope": scope,
        "excluded_relation_types": sorted(list(exclude_relation_types)) if exclude_relation_types else [],
        "included_relation_types": sorted(list(include_relation_types)) if include_relation_types else [],
        "stats": stats.__dict__,
        "top_rows": top_rows,
        "all_rows": rows,
    }



In [63]:
excluded = {"speaks_to", "favorable_toward", "unfavorable_toward"}

selected = filter_nodes_by_centrality(
        G,
        # node_type="Character",
        metric="betweenness",
        threshold=0,
        scope="global",
        exclude_relation_types=excluded,
    )
print(len(selected))

77


In [64]:
report = export_centrality_report(
    G,
    # node_type="Character",
    metric="betweenness",
    top_k=50,
    exclude_relation_types=excluded,
    bins=40,
    show_plot=False,
)
print(report["stats"])

{'count': 160, 'min': 0.0, 'max': 0.016752572109808933, 'mean': 0.0002314733350427243, 'median': 0.0, 'stdev': 0.0014766716152758094, 'p5': 0.0, 'p25': 0.0, 'p75': 8.392204648945688e-06, 'p95': 0.0005012604392032287}


In [8]:
from typing import Any, Dict, List, Optional, Tuple, Literal, Set
import re

Lang = Literal["en", "zh", "auto"]

def word_len(s: str, lang: Lang = "auto") -> int:
    s = s.strip()
    if not s:
        return 0

    if lang == "en":
        # 英文：按 word boundary
        return len(re.findall(r"\b[A-Za-z0-9]+\b", s))

    if lang == "zh":
        # 中文：按汉字（不做分词）
        return len(re.findall(r"[\u4e00-\u9fff]", s))

    # ---------- auto 模式 ----------
    # 中文字符数
    zh_count = len(re.findall(r"[\u4e00-\u9fff]", s))
    # 英文单词数
    en_count = len(re.findall(r"\b[A-Za-z0-9]+\b", s))

    # 如果明显是中文为主
    if zh_count > en_count:
        return zh_count

    # 否则按英文
    return en_count

In [9]:
word_len("\n".join(context))

8394

In [10]:
word_len(G.nodes["ent_bda73fd59e19"]["description"])

1190

In [11]:
from core.model_providers.openai_llm import OpenAILLM
from core import KAGConfig
import json
import re


config = KAGConfig.from_yaml("configs/config_openai.yaml")
llm = OpenAILLM(config)

/vepfs-mlp2/c20250513/241404044/users/roytian/anaconda3/envs/graphweaver/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
node_id = "ent_bda73fd59e19"
name = G.nodes[node_id]["name"]
entity_type = G.nodes[node_id]["type"]

In [13]:
with open("core/schema/default_entity_schema.json", "r") as f:
    entity_schema = json.load(f)

In [14]:
type2properties = {ent["type"]: ent["properties"] for ent in entity_schema}

In [15]:
type2description = {ent["type"]: ent["description"] for ent in entity_schema}

In [16]:
type2properties[entity_type]

{'name': 'Canonical name or form of address for this character node.',
 'gender': 'Gender (optional, if explicitly stated).',
 'age': 'Age or age range (optional, if explicitly stated).',
 'identity': 'Identity/occupation (optional, if explicitly stated).',
 'affiliation': 'Affiliated organization/faction (optional; should reference a Concept if extracted).'}

In [17]:
full_text = G.nodes["ent_bda73fd59e19"]["description"] + "\n".join(context)

In [18]:
word_len(full_text[:5000])

855

In [19]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
            chunk_size=1200, chunk_overlap=0,  length_function=word_len
        )

In [20]:
chunks = splitter.split_text(full_text)

In [21]:
word_len(chunks[-1])

93

In [22]:
chunks[0]

"A young woman with delicate features and dark eyes who opens the door for Ross and Berlin, appearing shy during introductions.\nA woman being questioned about her memories of spending time with Amber on the afternoon she left, appearing nervous during the conversation.\nA blind woman who recalls a man urging her to hurry due to snow and becomes uncomfortable when Berlin examines her hands.\nA woman who walks through the corridor unaware of the tension between Ross and Berlin and answers a question about her teaching subject, music composition and cello.\nA woman rehearsing music in an empty gymnasium, playing Elgar, who becomes aware of Berlin's presence and interacts with him.\nA woman providing testimony or recounting an observation about a car's sound, engaging in dialogue with Berlin.\nA woman who remains at the door, engages in conversation with Berlin, recalls someone from her past, and eventually leaves to get her coat.\nA woman who is reading a Braille volume in the deserted f

In [23]:
from core.builder.manager.information_manager import InformationExtractor

information_manager = InformationExtractor(config, llm)

In [24]:
full_description = ""
full_properties = dict()
for chunk in chunks:
    result = information_manager.extract_entity_properties(property_schema=type2description[entity_type]+"\n"+json.dumps(type2properties[entity_type], ensure_ascii=False, indent=2), text=chunk, entity_name=name)
    result = json.loads(result)
    properties = result["properties"]
    description = result["new_description"]
    full_description += description
    full_properties.update(properties)

In [25]:
len(list(full_properties.keys()))

28

In [26]:
len(chunks)

9

In [31]:
result = information_manager.merge_entity_properties(entity_name=name, 
                                                     full_description=full_description, 
                                                     properties=json.dumps(full_properties, ensure_ascii=False, indent=2),
                                                     num_properties=10)

In [67]:
json.loads(result)["properties"]

{'vision_status': 'blind',
 'profession': 'music teacher; music composition',
 'teaching_specialty': 'cello',
 'disability_origin': "father's alcoholism",
 'personality_trait': 'perceptive; reflective; emotionally expressive',
 'emotional_disposition': 'resilient; attentive',
 'relationships_summary': 'intimate partner of Berlin; cared for by Margie; niece of her aunt; close friend of Amber',
 'literary_interest': 'Hamlet; poetry',
 'daily_routine': 'manages household tasks; practices music; reads Braille',
 'personal_habits': 'grooms herself; prepares meals; maintains personal space'}

In [33]:
len(list(final_properties.keys()))

10

In [66]:
json.loads(result)["new_description"]

"Helena Robertson is a blind musician and music teacher who specializes in cello and music composition. She lives independently in her apartment, maintaining a structured daily routine and engaging in artistic practice, including reading Braille and playing guitar. Her identity is shaped by emotional resilience, perceptiveness, and a deep connection to literature and poetry. She shares a close, intimate relationship with Berlin, with whom she has a bond of mutual care and trust, and she is also supported by Margie and connected to her aunt as a niece. Despite enduring personal loss and the lasting impact of her father's alcoholism, she remains emotionally expressive and attentive to those around her. Her life reflects both vulnerability and agency within a complex emotional and social environment."